
# V50 LIVE — BanTH Target-Category Transfer for `identity_attack`

**Protected public-LB champion:** V38 = **0.608**  
**Minimum goal:** **0.609**  
**Main target:** **0.620**

V50 implements the single hypothesis selected by the final Astra resource audit:

> **Intermediate supervised adaptation on BanTH target categories, followed by matched competition-only fine-tuning for `identity_attack`.**

This is intentionally different from V43/V47. The novelty is **new external target-category supervision**, not another generic encoder, reranker, threshold search, or regularizer.

## External source

- Dataset: `aplycaebous/BanTH`
- Public card: MIT
- This notebook pins the immutable Hugging Face auto-converted Parquet snapshot:
  `ee3a295d27514e2e2f21ed5cd59fbff9198f5696`
- Source training input: original Romanized `Text`
- Source outputs:
  `Political`, `Religious`, `Gender`, `Personal Offense`,
  `Abusive/Violence`, `Origin`, `Body Shaming`
- `Misc` is excluded.
- BanTH labels are **never converted directly** into competition `identity_attack`.

## Matched experiment

**CONTROL**
- pinned `aplycaebous/tb-BanglaBERT-fpt`
- fresh one-logit identity head
- competition fitting rows only

**TREATMENT**
- same pinned encoder
- one epoch of BanTH seven-target adaptation
- source head discarded
- same fresh one-logit identity head initialization as control
- identical target-stage data/order/optimizer/epochs

Only the external supervised adaptation differs.

## Evaluation

- primary comparison uses only the 3297 LIVE train rows
- 3 fresh `StratifiedGroupKFold` outer folds, seed **5050**
- normalized duplicate/conflicting texts remain in one group
- inside each outer-training portion, grouped 5-fold split with seed `5150 + outer_fold`
- inner fold 0 is calibration; remaining groups are fitting
- identity threshold chosen only on calibration rows from `0.05..0.95` by `0.01`
- old 500 validation labels opened only after primary results are frozen
- 10,000 paired group bootstrap resamples
- all six other V38 outputs remain protected
- no Kaggle challenger is generated automatically

## V50 promotion gate

All must pass:

- mean outer identity F1 gain >= **+0.015**
- mean official-score gain >= **+0.00125**
- at least **2/3** strict fold wins
- worst fold F1 change >= **-0.010**
- pooled OOF F1 gain >= **+0.012**
- mean AP change >= 0
- worst fold AP change >= **-0.010**
- bootstrap positive fraction >= **90%**
- bootstrap 10th percentile of pooled official delta > 0
- eligible script subgroup F1 loss no worse than **-0.020**
- historical-500 veto: treatment identity F1 >= frozen V38 identity F1
- zero group leakage
- runtime <= five GPU hours

Passing means only: **eligible for a separate V38 integration check**.

At the end, upload `v50_results.zip` to ChatGPT.


In [ ]:

# 1. Upload V50 LIVE bundle

from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v50_LIVE_BanTH_identity_data.zip" in uploaded

DATA_DIR = pathlib.Path("/content/v50_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v50_LIVE_BanTH_identity_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install runtime dependencies

import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn>=1.4",
    "scipy",
    "safetensors",
    "huggingface_hub",
    "pyarrow",
    "tqdm",
])

print("✅ dependencies installed")


In [ ]:

# 3. Imports + frozen V50 configuration

import gc, re, os, time, math, json, random, hashlib, unicodedata, platform, zlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm

import sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

TARGET = "identity_attack"

MODEL_REPO = "aplycaebous/tb-BanglaBERT-fpt"
MODEL_REVISION = "0b24caaab62d12457ed0eb00ff0f459a1ea509bc"
EXPECTED_MODEL_WEIGHT_SHA256 = "9865ebfadc6f38ca385913d64bfa35eadfa03805aeefe42d8465ca075b353a78"

BANTH_REPO = "aplycaebous/BanTH"
BANTH_REVISION = "ee3a295d27514e2e2f21ed5cd59fbff9198f5696"
BANTH_TRAIN_FILE = "default/train/0000.parquet"

SOURCE_COLUMNS = [
    "Political",
    "Religious",
    "Gender",
    "Personal Offense",
    "Abusive/Violence",
    "Origin",
    "Body Shaming",
]

OUTER_SEED = 5050
INNER_SEED_BASE = 5150
SOURCE_SEED = 5000
PAIR_SEEDS = [5051, 5052, 5053]

N_OUTER = 3

MAX_LENGTH = 128
BATCH = 16
EVAL_BATCH = 64
ACCUM = 2

SOURCE_EPOCHS = 1
SOURCE_LR = 1e-5

TARGET_EPOCHS = 3
TARGET_LR = 2e-5

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0

THRESH_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)

NEAR_JACCARD = 0.90
NEAR_LENGTH_RATIO = 0.80
MIN_NEAR_CHARS = 20

PROTECTED_PUBLIC_LB = 0.608
MIN_PUBLIC_LB_GOAL = 0.609
MAIN_PUBLIC_LB_GOAL = 0.620

CHECKPOINT_EVERY_OPT_STEPS = 50

from google.colab import drive

DRIVE_OK = False
try:
    drive.mount("/content/drive", force_remount=False)
    DRIVE_OK = True
except Exception as e1:
    print("First Drive mount failed:", repr(e1))
    try:
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount("/content/drive", force_remount=True)
        DRIVE_OK = True
    except Exception as e2:
        print("Drive remount failed:", repr(e2))

if DRIVE_OK:
    RUN_DIR = Path("/content/drive/MyDrive/V50_BanTH_Identity")
else:
    print("Drive unavailable; using local /content. Completed caches will not survive runtime reset.")
    RUN_DIR = Path("/content/v50_run")

RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/content/v50_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(PAIR_SEEDS[0])

assert torch.cuda.is_available(), "Enable a GPU runtime (T4 preferred)."

print("GPU:", torch.cuda.get_device_name(0))
print("RUN_DIR:", RUN_DIR)


In [ ]:

# 4. Verify exact LIVE partition, hashes, and protected V38

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid500 = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2v = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v38_champion.csv", dtype={"ID":str})
oldb = np.load(DATA_DIR/"oldb.npz")

with open(DATA_DIR/"v50_manifest.json") as f:
    supplied_manifest = json.load(f)

assert len(train) == 3297
assert len(valid500) == 500
assert len(test) == 1000

assert train.ID.is_unique
assert valid500.ID.is_unique
assert test.ID.is_unique

assert not (set(train.ID) & set(valid500.ID))
assert not (set(train.ID) & set(test.ID))
assert not (set(valid500.ID) & set(test.ID))

assert valid500.ID.tolist() == v2v.ID.tolist()
assert test.ID.tolist() == v2t.ID.tolist()
assert test.ID.tolist() == champ.ID.tolist()

assert test.ID.head(4).tolist() == [
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda:f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

for name in [
    "train.csv","validation.csv","test.csv",
    "v2_valid.csv","v2_test.csv","oldb.npz",
    "submission_v38_champion.csv",
]:
    got = file_sha256(DATA_DIR/name)
    expected = supplied_manifest["files"][name]["sha256"]
    assert got == expected, f"Hash mismatch: {name}"

print("✅ exact LIVE partition verified")
print("✅ data hashes verified")
print("✅ protected V38 verified")
print("V38 positive counts:")
print(champ[[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]].sum().to_dict())


In [ ]:

# 5. Download and verify the exact tb-BanglaBERT checkpoint used in V48/V49

MODEL_DIR = Path("/content/tb_banglabert_fpt_pinned")

if not MODEL_DIR.exists() or not any(MODEL_DIR.iterdir()):
    snapshot_download(
        repo_id=MODEL_REPO,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )

weight_candidates = [
    MODEL_DIR/"pytorch_model.bin",
    MODEL_DIR/"model.safetensors",
]
existing = [p for p in weight_candidates if p.exists()]
assert existing, "Pinned model has no visible weight file."

weight_hashes = {p.name:file_sha256(p) for p in existing}
assert EXPECTED_MODEL_WEIGHT_SHA256 in weight_hashes.values(), (
    "Pinned model weights do not match the V48/V49 recorded hash."
)

print("Model weight hashes:")
print(json.dumps(weight_hashes, indent=2))
print("✅ exact model revision + weight hash verified")


In [ ]:

# 6. Build competition text groups and freeze fresh folds BEFORE external labels are used

BN_RE = re.compile(r"[\u0980-\u09FF]")
LAT_RE = re.compile(r"[A-Za-z]")

def model_text(x):
    x = unicodedata.normalize("NFC", str(x))
    x = x.replace("\u200c"," ").replace("\u200d"," ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def canonical_text(x):
    return model_text(x).casefold()

def script_type(x):
    s = str(x)
    bn = bool(BN_RE.search(s))
    lat = bool(LAT_RE.search(s))
    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "no_bengali"

for df in [train, valid500, test]:
    df["model_text"] = df.text.map(model_text)
    df["group_key"] = df.text.map(canonical_text)
    df["script"] = df.model_text.map(script_type)

sgkf = StratifiedGroupKFold(
    n_splits=N_OUTER,
    shuffle=True,
    random_state=OUTER_SEED,
)

train["outer_fold"] = -1

for fold, (_, oi) in enumerate(
    sgkf.split(
        train,
        y=train[TARGET].to_numpy(int),
        groups=train.group_key,
    )
):
    train.loc[train.index[oi], "outer_fold"] = fold

assert (train.outer_fold >= 0).all()
assert train.groupby("group_key").outer_fold.nunique().max() == 1

inner_maps = {}
split_rows = []

for fold in range(N_OUTER):
    outer_mask = train.outer_fold == fold
    outer_train = train.loc[~outer_mask].copy()

    inner = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=INNER_SEED_BASE + fold,
    )

    fit_local, calib_local = next(
        inner.split(
            outer_train,
            y=outer_train[TARGET].to_numpy(int),
            groups=outer_train.group_key,
        )
    )

    fit_idx = outer_train.index.to_numpy()[fit_local]
    calib_idx = outer_train.index.to_numpy()[calib_local]
    outer_idx = train.index.to_numpy()[outer_mask.to_numpy()]

    fit_groups = set(train.loc[fit_idx, "group_key"])
    calib_groups = set(train.loc[calib_idx, "group_key"])
    outer_groups = set(train.loc[outer_idx, "group_key"])

    assert fit_groups.isdisjoint(calib_groups)
    assert fit_groups.isdisjoint(outer_groups)
    assert calib_groups.isdisjoint(outer_groups)

    inner_maps[fold] = {
        "fit_idx": fit_idx,
        "calib_idx": calib_idx,
        "outer_idx": outer_idx,
    }

    for role, idx in [
        ("fit", fit_idx),
        ("calib", calib_idx),
        ("outer", outer_idx),
    ]:
        d = train.loc[idx]
        split_rows.append({
            "fold": fold,
            "role": role,
            "rows": len(d),
            "groups": d.group_key.nunique(),
            "identity_pos": int(d[TARGET].sum()),
            "identity_neg": int((1-d[TARGET]).sum()),
            "native": int((d.script=="native").sum()),
            "mixed": int((d.script=="mixed").sum()),
            "no_bengali": int((d.script=="no_bengali").sum()),
        })

split_report = pd.DataFrame(split_rows)
display(split_report)
split_report.to_csv(EXPORT_DIR/"v50_split_report.csv", index=False)

print("✅ folds frozen before BanTH labels are accessed")
print("✅ zero competition group leakage")


In [ ]:

# 7. Pin BanTH immutable snapshot, verify MIT metadata, and load TEXT VIEWS ONLY
#
# IMPORTANT:
# This cell does not read source labels. Overlap filtering is frozen first.

api = HfApi()

# Current author-hosted card must still declare MIT.
main_info = api.dataset_info(BANTH_REPO)
card = getattr(main_info, "card_data", None)
license_value = None
if card is not None:
    license_value = getattr(card, "license", None)
    if license_value is None and isinstance(card, dict):
        license_value = card.get("license")

if isinstance(license_value, (list,tuple)):
    license_norm = " ".join(map(str,license_value)).lower()
else:
    license_norm = str(license_value).lower()

assert "mit" in license_norm, (
    f"BanTH license metadata is not MIT as expected: {license_value!r}"
)

# Download the exact immutable auto-converted parquet snapshot.
banth_path = hf_hub_download(
    repo_id=BANTH_REPO,
    repo_type="dataset",
    revision=BANTH_REVISION,
    filename=BANTH_TRAIN_FILE,
)

banth_file_hash = file_sha256(banth_path)

banth_all = pd.read_parquet(banth_path)

EXPECTED_COLUMNS = {
    "Text","Label","Political","Religious","Gender",
    "Personal Offense","Abusive/Violence","Origin",
    "Body Shaming","Misc","bangla","english"
}

assert EXPECTED_COLUMNS.issubset(set(banth_all.columns)), (
    "BanTH schema differs from the audited viewer. "
    f"Observed: {banth_all.columns.tolist()}"
)

assert len(banth_all) == 29879, (
    f"Expected BanTH train split 29,879 rows, got {len(banth_all)}"
)

# Text-only view. No target labels are touched yet.
banth_text = banth_all[["Text","bangla","english"]].copy()

print("BanTH pinned revision:", BANTH_REVISION)
print("BanTH train parquet SHA256:", banth_file_hash)
print("BanTH train rows:", len(banth_text))
print("BanTH license metadata:", license_value)
print("✅ immutable BanTH text snapshot loaded")


In [ ]:

# 8. Text-only BanTH -> competition overlap filtering
#
# Stages:
# 1) exact raw
# 2) exact canonical (NFC + casefold + whitespace + zero-width removal)
# 3) conservative exact char-5-gram Jaccard >= .90, length ratio >= .80
#
# Strings shorter than 20 chars use exact/canonical matching only.
#
# Alternate BanTH views (bangla/english) are used ONLY for exclusion.
# No external labels are read in this cell.

competition_text = pd.concat([
    train[["ID","text"]].assign(split="train"),
    valid500[["ID","text"]].assign(split="validation"),
    test[["ID","text"]].assign(split="test"),
], ignore_index=True)

competition_text["raw"] = competition_text.text.astype(str)
competition_text["canon"] = competition_text.text.map(canonical_text)

raw_to_splits = defaultdict(set)
canon_to_splits = defaultdict(set)

for r in competition_text.itertuples(index=False):
    raw_to_splits[r.raw].add(r.split)
    canon_to_splits[r.canon].add(r.split)

# Unique competition canonical strings for near-duplicate matching.
comp_unique = (
    competition_text[["canon","split"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

def gram5_set(s):
    s = str(s)
    if len(s) < 5:
        return {s}
    return {s[i:i+5] for i in range(len(s)-4)}

def token_order(tok):
    # Stable pseudo-random global order; prefix filtering works under any
    # shared total token order. CRC32 + lexical tie-break is deterministic.
    return (zlib.crc32(tok.encode("utf-8")) & 0xffffffff, tok)

def prefix_tokens(tokens, threshold=NEAR_JACCARD):
    toks = sorted(tokens, key=token_order)
    p = len(toks) - math.ceil(threshold*len(toks)) + 1
    p = max(1, min(len(toks), p))
    return toks[:p]

comp_grams = []
prefix_index = defaultdict(list)

for i, s in enumerate(comp_unique.canon.astype(str)):
    gs = gram5_set(s)
    comp_grams.append(gs)

    if len(s) >= MIN_NEAR_CHARS:
        for tok in prefix_tokens(gs):
            prefix_index[tok].append(i)

def near_match_splits(canon):
    if len(canon) < MIN_NEAR_CHARS:
        return set()

    gs = gram5_set(canon)
    candidates = set()

    for tok in prefix_tokens(gs):
        candidates.update(prefix_index.get(tok, []))

    found = set()

    for ci in candidates:
        other = comp_unique.iloc[ci].canon
        ratio = min(len(canon),len(other))/max(len(canon),len(other),1)

        if ratio < NEAR_LENGTH_RATIO:
            continue

        og = comp_grams[ci]
        union = len(gs | og)
        if union == 0:
            continue

        jac = len(gs & og)/union

        if jac >= NEAR_JACCARD:
            found.add(comp_unique.iloc[ci].split)

    return found

stage = np.array(["keep"]*len(banth_text), dtype=object)
match_splits = [set() for _ in range(len(banth_text))]

view_cols = ["Text","bangla","english"]

# Stage 1: raw exact.
for i, row in enumerate(tqdm(
    banth_text[view_cols].itertuples(index=False, name=None),
    total=len(banth_text),
    desc="BanTH raw exact"
)):
    splits=set()

    for v in row:
        if pd.isna(v):
            continue
        s=str(v)
        if s in raw_to_splits:
            splits |= raw_to_splits[s]

    if splits:
        stage[i]="raw_exact"
        match_splits[i]=splits

# Stage 2: canonical exact.
for i, row in enumerate(tqdm(
    banth_text[view_cols].itertuples(index=False, name=None),
    total=len(banth_text),
    desc="BanTH canonical exact"
)):
    if stage[i] != "keep":
        continue

    splits=set()

    for v in row:
        if pd.isna(v):
            continue
        s=canonical_text(v)
        if s in canon_to_splits:
            splits |= canon_to_splits[s]

    if splits:
        stage[i]="canonical_exact"
        match_splits[i]=splits

# Stage 3: conservative near duplicates.
for i, row in enumerate(tqdm(
    banth_text[view_cols].itertuples(index=False, name=None),
    total=len(banth_text),
    desc="BanTH near duplicate"
)):
    if stage[i] != "keep":
        continue

    splits=set()

    for v in row:
        if pd.isna(v):
            continue

        s=canonical_text(v)

        if len(s) >= MIN_NEAR_CHARS:
            splits |= near_match_splits(s)

        if splits:
            break

    if splits:
        stage[i]="near_duplicate"
        match_splits[i]=splits

# Remove matching external TEXT GROUPS wholesale.
source_group = banth_text.Text.map(canonical_text).to_numpy(object)
matched_groups = set(source_group[stage!="keep"])

propagated = (stage=="keep") & np.isin(source_group, list(matched_groups))
stage[propagated] = "group_propagated"

keep_mask = stage=="keep"

overlap_rows=[]

for st in ["raw_exact","canonical_exact","near_duplicate","group_propagated"]:
    mask=stage==st

    rec={
        "stage":st,
        "rows":int(mask.sum()),
    }

    # Split counts are direct matches only; propagated groups are counted
    # separately without exposing text or IDs.
    for sp in ["train","validation","test"]:
        rec[sp+"_direct_rows"] = int(sum(
            (st != "group_propagated")
            and mask[i]
            and (sp in match_splits[i])
            for i in range(len(stage))
        ))

    overlap_rows.append(rec)

overlap_summary = pd.DataFrame(overlap_rows)

print("Starting BanTH train rows:", len(banth_text))
print("Retained after text-only filtering:", int(keep_mask.sum()))
print("Excluded union:", int((~keep_mask).sum()))
display(overlap_summary)

overlap_summary.to_csv(
    EXPORT_DIR/"v50_banth_overlap_summary.csv",
    index=False
)

np.save(
    EXPORT_DIR/"v50_banth_keep_mask.npy",
    keep_mask
)

print("✅ external overlap filtering frozen BEFORE source labels are used")


In [ ]:

# 9. NOW access BanTH source labels and run source-data preflight

banth = banth_all.loc[keep_mask].copy().reset_index(drop=True)

# Validate label semantics: only 0/1/missing accepted.
for c in SOURCE_COLUMNS:
    numeric = pd.to_numeric(banth[c], errors="coerce")
    original_nonmissing = banth[c].notna()

    # Any nonmissing value that did not coerce is an unknown semantic.
    if ((numeric.isna()) & original_nonmissing).any():
        bad = banth.loc[(numeric.isna()) & original_nonmissing, c].head().tolist()
        raise ValueError(f"Unexpected nonnumeric values in {c}: {bad}")

    vals = set(numeric.dropna().astype(int).unique().tolist())
    if not vals.issubset({0,1}):
        raise ValueError(f"Unexpected label values in {c}: {vals}")

    banth[c] = numeric

banth["source_group"] = banth.Text.map(canonical_text)

# Distinct positive source groups after filtering.
grouped_source = banth.groupby("source_group")[SOURCE_COLUMNS].max()

positive_group_counts = {
    c:int(grouped_source[c].fillna(0).eq(1).sum())
    for c in SOURCE_COLUMNS
}

print("BanTH positive groups after filtering:")
print(json.dumps(positive_group_counts,indent=2))

for critical in ["Religious","Gender","Origin"]:
    assert positive_group_counts[critical] >= 50, (
        f"Preflight stop: only {positive_group_counts[critical]} "
        f"distinct positive groups remain for {critical}."
    )

# Missing labels remain masked, never filled with zero.
source_y = banth[SOURCE_COLUMNS].to_numpy(np.float32)
source_mask = np.isfinite(source_y).astype(np.float32)

# Placeholder values are used only where mask=0; masked entries contribute 0 loss.
source_y_safe = np.nan_to_num(source_y, nan=0.0)

pos_weights=[]

for j,c in enumerate(SOURCE_COLUMNS):
    known = source_mask[:,j] > 0
    yj = source_y_safe[known,j]

    pos=float((yj==1).sum())
    neg=float((yj==0).sum())

    assert pos > 0 and neg > 0, f"Degenerate source label: {c}"

    pos_weights.append(
        min(5.0, math.sqrt(neg/pos))
    )

SOURCE_POS_WEIGHT = np.asarray(pos_weights,np.float32)

source_distribution = pd.DataFrame({
    "label":SOURCE_COLUMNS,
    "known":[int(source_mask[:,j].sum()) for j in range(len(SOURCE_COLUMNS))],
    "positive":[int(((source_y_safe[:,j]==1)&(source_mask[:,j]>0)).sum()) for j in range(len(SOURCE_COLUMNS))],
    "positive_groups":[positive_group_counts[c] for c in SOURCE_COLUMNS],
    "pos_weight":SOURCE_POS_WEIGHT,
})

display(source_distribution)
source_distribution.to_csv(
    EXPORT_DIR/"v50_banth_source_distribution.csv",
    index=False
)

print("✅ BanTH label/schema/count preflight passed")


In [ ]:

# 10. Tokenizer, datasets, fixed-order sampler and metric helpers

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=True)

class SourceDS(Dataset):
    def __init__(self,df,y,mask):
        self.text=df.Text.astype(str).map(model_text).tolist()
        self.y=np.asarray(y,np.float32)
        self.mask=np.asarray(mask,np.float32)

    def __len__(self):
        return len(self.text)

    def __getitem__(self,i):
        item=tokenizer(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        item["labels"]=self.y[i]
        item["label_mask"]=self.mask[i]
        return item

class TargetDS(Dataset):
    def __init__(self,df,with_labels=True):
        self.text=df.model_text.astype(str).tolist()
        self.y=df[TARGET].to_numpy(np.float32) if with_labels else None

    def __len__(self):
        return len(self.text)

    def __getitem__(self,i):
        item=tokenizer(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            item["labels"]=float(self.y[i])
        return item

class SourceCollator:
    def __call__(self,items):
        y=torch.tensor(np.stack([x.pop("labels") for x in items]),dtype=torch.float32)
        m=torch.tensor(np.stack([x.pop("label_mask") for x in items]),dtype=torch.float32)

        b=tokenizer.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        b["labels"]=y
        b["label_mask"]=m
        return b

class TargetCollator:
    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                [x.pop("labels") for x in items],
                dtype=torch.float32
            )

        b=tokenizer.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        if y is not None:
            b["labels"]=y
        return b

source_collator=SourceCollator()
target_collator=TargetCollator()

class FixedOrderSampler(Sampler):
    def __init__(self,order):
        self.order=list(map(int,order))
    def __iter__(self):
        return iter(self.order)
    def __len__(self):
        return len(self.order)

def f1_pos(y,p):
    return f1_score(y,p,pos_label=1,zero_division=0)

def choose_threshold(y,prob):
    rows=[]
    for t in THRESH_GRID:
        pred=(prob>=t).astype(int)
        score=f1_pos(y,pred)
        rows.append((score,-abs(float(t)-0.5),float(t)))
    rows.sort(reverse=True)
    return rows[0][2],rows[0][0]

def safe_ap(y,p):
    return average_precision_score(y,p)

def logit_np(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def calibrated_margin(prob,threshold):
    return logit_np(prob)-logit_np([threshold])[0]

@torch.no_grad()
def predict(model,loader):
    model.eval()
    out=[]

    for b in loader:
        b.pop("labels",None)
        b.pop("label_mask",None)

        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp).logits

        if logits.ndim==2 and logits.shape[1]==1:
            logits=logits.squeeze(-1)

        out.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

    return np.concatenate(out)


In [ ]:

# 11. Checkpoint/RNG helpers

def json_hash(obj):
    return hashlib.sha256(
        json.dumps(obj,sort_keys=True).encode()
    ).hexdigest()

CONFIG_CORE = {
    "model_repo":MODEL_REPO,
    "model_revision":MODEL_REVISION,
    "model_weight_hash":EXPECTED_MODEL_WEIGHT_SHA256,
    "banth_repo":BANTH_REPO,
    "banth_revision":BANTH_REVISION,
    "banth_train_sha256":banth_file_hash,
    "outer_seed":OUTER_SEED,
    "inner_seed_base":INNER_SEED_BASE,
    "source_seed":SOURCE_SEED,
    "pair_seeds":PAIR_SEEDS,
    "source_columns":SOURCE_COLUMNS,
    "source_epochs":SOURCE_EPOCHS,
    "source_lr":SOURCE_LR,
    "target_epochs":TARGET_EPOCHS,
    "target_lr":TARGET_LR,
    "batch":BATCH,
    "accum":ACCUM,
    "max_length":MAX_LENGTH,
    "weight_decay":WEIGHT_DECAY,
    "warmup_ratio":WARMUP_RATIO,
    "threshold_grid":THRESH_GRID.tolist(),
    "train_hash":supplied_manifest["files"]["train.csv"]["sha256"],
}

CONFIG_FP=json_hash(CONFIG_CORE)

def capture_rng():
    return {
        "python":random.getstate(),
        "numpy":np.random.get_state(),
        "torch_cpu":torch.get_rng_state(),
        "torch_cuda":torch.cuda.get_rng_state_all(),
    }

def restore_rng(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch_cpu"])
    torch.cuda.set_rng_state_all(s["torch_cuda"])

def save_ckpt(path,model,opt,sched,scaler,epoch,next_batch,opt_steps,extra=None):
    torch.save({
        "config_fp":CONFIG_FP,
        "model":model.state_dict(),
        "optimizer":opt.state_dict(),
        "scheduler":sched.state_dict(),
        "scaler":scaler.state_dict(),
        "epoch":int(epoch),
        "next_batch":int(next_batch),
        "opt_steps":int(opt_steps),
        "rng":capture_rng(),
        "extra":extra or {},
    },path)

def load_ckpt(path,model,opt,sched,scaler):
    ck=torch.load(path,map_location="cpu",weights_only=False)
    assert ck["config_fp"]==CONFIG_FP, "Checkpoint/config mismatch."

    model.load_state_dict(ck["model"])
    opt.load_state_dict(ck["optimizer"])
    sched.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    restore_rng(ck["rng"])

    return ck


In [ ]:

# 12. Build source/target models

def build_source_model(seed):
    seed_all(seed)
    return AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR),
        num_labels=len(SOURCE_COLUMNS),
        problem_type="multi_label_classification",
        ignore_mismatched_sizes=True,
    )

def build_target_model(seed,adapted_encoder_state=None):
    seed_all(seed)

    model=AutoModelForSequenceClassification.from_pretrained(
        str(MODEL_DIR),
        num_labels=1,
        problem_type="multi_label_classification",
        ignore_mismatched_sizes=True,
    )

    if adapted_encoder_state is not None:
        assert hasattr(model,"electra"), "Expected ELECTRA-family model."
        model.electra.load_state_dict(
            adapted_encoder_state,
            strict=True,
        )

    return model

def adamw_all(model,lr):
    return torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

def masked_source_loss(logits,y,mask,pos_weight):
    raw=F.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        reduction="none",
        pos_weight=pos_weight,
    )

    weighted=raw*mask
    denom=mask.sum().clamp_min(1.0)

    return weighted.sum()/denom


In [ ]:

# 13. 100-step cold throughput pilot BEFORE full GPU commitment

PILOT_STEPS=100

pilot_model=build_source_model(SOURCE_SEED).to("cuda")
pilot_opt=adamw_all(pilot_model,SOURCE_LR)
pilot_scaler=torch.amp.GradScaler("cuda")

pilot_order=np.random.default_rng(SOURCE_SEED).permutation(len(banth))

pilot_loader=DataLoader(
    SourceDS(banth,source_y_safe,source_mask),
    batch_size=BATCH,
    sampler=FixedOrderSampler(pilot_order),
    num_workers=0,
    pin_memory=True,
    collate_fn=source_collator,
)

posw=torch.tensor(SOURCE_POS_WEIGHT,dtype=torch.float32,device="cuda")

pilot_model.train()
pilot_opt.zero_grad(set_to_none=True)

torch.cuda.synchronize()
t0=time.time()
seen=0

for step,b in enumerate(pilot_loader):
    if step>=PILOT_STEPS:
        break

    y=b.pop("labels").to("cuda")
    m=b.pop("label_mask").to("cuda")
    inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

    with torch.autocast(device_type="cuda",dtype=torch.float16):
        logits=pilot_model(**inp).logits
        loss=masked_source_loss(logits,y,m,posw)

    pilot_scaler.scale(loss).backward()

    if (step+1)%ACCUM==0:
        pilot_scaler.unscale_(pilot_opt)
        torch.nn.utils.clip_grad_norm_(pilot_model.parameters(),GRAD_CLIP)
        pilot_scaler.step(pilot_opt)
        pilot_scaler.update()
        pilot_opt.zero_grad(set_to_none=True)

    seen+=1

torch.cuda.synchronize()
sec_per_batch=(time.time()-t0)/max(seen,1)

source_batches=math.ceil(len(banth)/BATCH)*SOURCE_EPOCHS

target_batches=0
for fold in range(N_OUTER):
    nfit=len(inner_maps[fold]["fit_idx"])
    target_batches += math.ceil(nfit/BATCH)*TARGET_EPOCHS*2

projected_training_min=(
    (source_batches+target_batches)*sec_per_batch/60.0
)

projected_total_min=projected_training_min*1.25 + 30.0

print(f"Pilot seconds/batch: {sec_per_batch:.3f}")
print(f"Projected GPU training: {projected_training_min:.1f} min")
print(f"Projected total with 25% contingency + 30 min: {projected_total_min:.1f} min")

del pilot_model,pilot_opt,pilot_scaler,pilot_loader
gc.collect()
torch.cuda.empty_cache()

assert projected_total_min <= 270.0, (
    "Preflight stop: projected V50 completion exceeds 4.5 hours. "
    "Do not shorten only one arm or silently omit folds."
)

print("✅ projected runtime fits the V50 budget")


In [ ]:

# 14. One-time BanTH seven-target intermediate adaptation

ADAPTED_ENCODER_PATH=RUN_DIR/"banth_adapted_electra.pt"
SOURCE_COMPLETE=RUN_DIR/"banth_source_complete.json"
SOURCE_CKPT=RUN_DIR/"banth_source_latest.pt"

if ADAPTED_ENCODER_PATH.exists() and SOURCE_COMPLETE.exists():
    adapted_encoder_state=torch.load(
        ADAPTED_ENCODER_PATH,
        map_location="cpu",
        weights_only=True,
    )
    print("✅ reuse completed BanTH-adapted encoder")
else:
    model=build_source_model(SOURCE_SEED).to("cuda")
    opt=adamw_all(model,SOURCE_LR)

    batches=math.ceil(len(banth)/BATCH)
    opt_steps_total=math.ceil(batches/ACCUM)*SOURCE_EPOCHS

    sched=get_linear_schedule_with_warmup(
        opt,
        int(opt_steps_total*WARMUP_RATIO),
        opt_steps_total,
    )

    scaler=torch.amp.GradScaler("cuda")

    start_epoch=1
    start_batch=0
    opt_steps=0

    if SOURCE_CKPT.exists():
        ck=load_ckpt(SOURCE_CKPT,model,opt,sched,scaler)
        start_epoch=ck["epoch"]
        start_batch=ck["next_batch"]
        opt_steps=ck["opt_steps"]
        print("↩ resume source:",start_epoch,start_batch,opt_steps)

    posw=torch.tensor(
        SOURCE_POS_WEIGHT,
        dtype=torch.float32,
        device="cuda"
    )

    t_source=time.time()

    for epoch in range(start_epoch,SOURCE_EPOCHS+1):
        epoch_seed=SOURCE_SEED + epoch*101

        if not (SOURCE_CKPT.exists() and epoch==start_epoch and start_batch>0):
            seed_all(epoch_seed)

        order=np.random.default_rng(epoch_seed).permutation(len(banth))

        loader=DataLoader(
            SourceDS(banth,source_y_safe,source_mask),
            batch_size=BATCH,
            sampler=FixedOrderSampler(order),
            num_workers=0,
            pin_memory=True,
            collate_fn=source_collator,
        )

        model.train()
        opt.zero_grad(set_to_none=True)

        resume_from=start_batch if epoch==start_epoch else 0

        for step,b in enumerate(tqdm(loader,desc=f"BanTH source e{epoch}/{SOURCE_EPOCHS}")):
            if step<resume_from:
                continue

            y=b.pop("labels").to("cuda")
            m=b.pop("label_mask").to("cuda")
            inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inp).logits
                loss=masked_source_loss(logits,y,m,posw)

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite BanTH source loss")

            scaler.scale(loss/ACCUM).backward()

            do_step=((step+1)%ACCUM==0) or ((step+1)==len(loader))

            if do_step:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),GRAD_CLIP
                )

                old=scaler.get_scale()
                scaler.step(opt)
                scaler.update()

                if scaler.get_scale()>=old:
                    sched.step()

                opt.zero_grad(set_to_none=True)
                opt_steps+=1

                if opt_steps%CHECKPOINT_EVERY_OPT_STEPS==0:
                    save_ckpt(
                        SOURCE_CKPT,model,opt,sched,scaler,
                        epoch,step+1,opt_steps
                    )

        save_ckpt(
            SOURCE_CKPT,model,opt,sched,scaler,
            epoch+1,0,opt_steps
        )

        start_batch=0

    adapted_encoder_state={
        k:v.detach().cpu()
        for k,v in model.electra.state_dict().items()
    }

    torch.save(adapted_encoder_state,ADAPTED_ENCODER_PATH)

    source_done={
        "elapsed_min":(time.time()-t_source)/60.0,
        "rows":len(banth),
        "source_columns":SOURCE_COLUMNS,
        "positive_groups":positive_group_counts,
        "pos_weights":SOURCE_POS_WEIGHT.tolist(),
        "banth_revision":BANTH_REVISION,
        "banth_train_sha256":banth_file_hash,
    }

    SOURCE_COMPLETE.write_text(json.dumps(source_done,indent=2))

    del model,opt,sched,scaler,loader
    gc.collect()
    torch.cuda.empty_cache()

    print("✅ BanTH source adaptation complete")


In [ ]:

# 15. Target training helpers: matched control vs BanTH-adapted treatment

def make_target_eval_loader(df,with_labels=True):
    return DataLoader(
        TargetDS(df.reset_index(drop=True),with_labels),
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=target_collator,
    )

def train_target_arm(fold,arm):
    assert arm in ["control","treatment"]

    seed=PAIR_SEEDS[fold]

    fit=train.loc[inner_maps[fold]["fit_idx"]].copy().reset_index(drop=True)
    calib=train.loc[inner_maps[fold]["calib_idx"]].copy().reset_index(drop=True)
    outer=train.loc[inner_maps[fold]["outer_idx"]].copy().reset_index(drop=True)

    complete=RUN_DIR/f"fold{fold}_{arm}_complete.npz"
    latest=RUN_DIR/f"fold{fold}_{arm}_latest.pt"
    history_path=RUN_DIR/f"fold{fold}_{arm}_history.csv"

    if complete.exists():
        print("✅ reuse",complete.name)
        with np.load(complete,allow_pickle=False) as z:
            return {k:z[k] for k in z.files}

    state=adapted_encoder_state if arm=="treatment" else None
    model=build_target_model(seed,state).to("cuda")

    opt=adamw_all(model,TARGET_LR)

    batches_per_epoch=math.ceil(len(fit)/BATCH)
    opt_steps_total=math.ceil(batches_per_epoch/ACCUM)*TARGET_EPOCHS

    sched=get_linear_schedule_with_warmup(
        opt,
        int(opt_steps_total*WARMUP_RATIO),
        opt_steps_total,
    )

    scaler=torch.amp.GradScaler("cuda")

    start_epoch=1
    start_batch=0
    opt_steps=0

    if latest.exists():
        ck=load_ckpt(latest,model,opt,sched,scaler)
        start_epoch=ck["epoch"]
        start_batch=ck["next_batch"]
        opt_steps=ck["opt_steps"]
        print("↩ resume",fold,arm,start_epoch,start_batch,opt_steps)

    calib_loader=make_target_eval_loader(calib,True)
    outer_loader=make_target_eval_loader(outer,True)
    valid_loader=make_target_eval_loader(valid500,True)
    test_loader=make_target_eval_loader(test,False)

    history=[]
    if history_path.exists():
        history=pd.read_csv(history_path).to_dict("records")

    arm_start=time.time()

    for epoch in range(start_epoch,TARGET_EPOCHS+1):
        epoch_seed=seed + epoch*101

        if not (latest.exists() and epoch==start_epoch and start_batch>0):
            seed_all(epoch_seed)

        # Identical row order across control/treatment.
        order=np.random.default_rng(epoch_seed).permutation(len(fit))

        loader=DataLoader(
            TargetDS(fit,True),
            batch_size=BATCH,
            sampler=FixedOrderSampler(order),
            num_workers=0,
            pin_memory=True,
            collate_fn=target_collator,
        )

        model.train()
        opt.zero_grad(set_to_none=True)

        running=0.0
        seen_rows=0

        resume_from=start_batch if epoch==start_epoch else 0

        for step,b in enumerate(
            tqdm(loader,desc=f"fold {fold} {arm} e{epoch}/{TARGET_EPOCHS}")
        ):
            if step<resume_from:
                continue

            y=b.pop("labels").to("cuda")
            inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inp).logits.squeeze(-1)
                loss=F.binary_cross_entropy_with_logits(
                    logits.float(),y
                )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite target loss fold={fold} arm={arm}"
                )

            scaler.scale(loss/ACCUM).backward()

            running += float(loss.item())*len(y)
            seen_rows += len(y)

            do_step=((step+1)%ACCUM==0) or ((step+1)==len(loader))

            if do_step:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),GRAD_CLIP
                )

                old=scaler.get_scale()
                scaler.step(opt)
                scaler.update()

                if scaler.get_scale()>=old:
                    sched.step()

                opt.zero_grad(set_to_none=True)
                opt_steps+=1

                if opt_steps%CHECKPOINT_EVERY_OPT_STEPS==0:
                    save_ckpt(
                        latest,model,opt,sched,scaler,
                        epoch,step+1,opt_steps
                    )

        history.append({
            "fold":fold,
            "arm":arm,
            "epoch":epoch,
            "mean_loss_since_resume":running/max(seen_rows,1),
            "optimizer_steps":opt_steps,
            "elapsed_min":(time.time()-arm_start)/60.0,
        })

        pd.DataFrame(history).to_csv(history_path,index=False)

        save_ckpt(
            latest,model,opt,sched,scaler,
            epoch+1,0,opt_steps
        )

        start_batch=0

    # Fixed final-epoch inference; no outer-label checkpoint selection.
    out={
        "calib_ids":calib.ID.astype(str).to_numpy(dtype=str),
        "outer_ids":outer.ID.astype(str).to_numpy(dtype=str),
        "calib_y":calib[TARGET].to_numpy(np.int8),
        "outer_y":outer[TARGET].to_numpy(np.int8),
        "calib_prob":predict(model,calib_loader).astype(np.float32),
        "outer_prob":predict(model,outer_loader).astype(np.float32),
        "valid500_prob":predict(model,valid_loader).astype(np.float32),
        "test_prob":predict(model,test_loader).astype(np.float32),
        "elapsed_min":np.array(
            [(time.time()-arm_start)/60.0],
            dtype=np.float32
        ),
    }

    np.savez_compressed(complete,**out)

    del model,opt,sched,scaler,loader,calib_loader,outer_loader,valid_loader,test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return out


In [ ]:

# 16. Execute all six target fits and freeze fold results

experiment_start=time.time()

raw={}
fold_rows=[]
threshold_control=[]
threshold_treatment=[]

for fold in range(N_OUTER):
    raw[fold]={}

    for arm in ["control","treatment"]:
        raw[fold][arm]=train_target_arm(fold,arm)

    c=raw[fold]["control"]
    t=raw[fold]["treatment"]

    assert c["calib_ids"].tolist()==t["calib_ids"].tolist()
    assert c["outer_ids"].tolist()==t["outer_ids"].tolist()
    assert np.array_equal(c["calib_y"],t["calib_y"])
    assert np.array_equal(c["outer_y"],t["outer_y"])

    tc,_=choose_threshold(c["calib_y"],c["calib_prob"])
    tt,_=choose_threshold(t["calib_y"],t["calib_prob"])

    threshold_control.append(tc)
    threshold_treatment.append(tt)

    pc=(c["outer_prob"]>=tc).astype(int)
    pt=(t["outer_prob"]>=tt).astype(int)
    y=c["outer_y"].astype(int)

    f1c=f1_pos(y,pc)
    f1t=f1_pos(y,pt)

    apc=safe_ap(y,c["outer_prob"])
    apt=safe_ap(y,t["outer_prob"])

    fold_rows.append({
        "fold":fold,
        "control_threshold":tc,
        "treatment_threshold":tt,
        "control_f1":f1c,
        "treatment_f1":f1t,
        "f1_gain":f1t-f1c,
        "official_gain":(f1t-f1c)/12.0,
        "control_ap":apc,
        "treatment_ap":apt,
        "ap_gain":apt-apc,
        "control_precision":precision_score(y,pc,zero_division=0),
        "treatment_precision":precision_score(y,pt,zero_division=0),
        "control_recall":recall_score(y,pc,zero_division=0),
        "treatment_recall":recall_score(y,pt,zero_division=0),
        "true_pos":int(y.sum()),
        "control_pred_pos":int(pc.sum()),
        "treatment_pred_pos":int(pt.sum()),
    })

fold_report=pd.DataFrame(fold_rows)
display(fold_report.round(6))
fold_report.to_csv(EXPORT_DIR/"v50_fold_results.csv",index=False)

print("Six target fits elapsed min:",(time.time()-experiment_start)/60.0)


In [ ]:

# 17. Pooled OOF + script groups

control_oof=np.full(len(train),np.nan,np.float32)
treatment_oof=np.full(len(train),np.nan,np.float32)
control_pred=np.full(len(train),-1,np.int8)
treatment_pred=np.full(len(train),-1,np.int8)
fold_col=np.full(len(train),-1,np.int8)

id_to_idx={x:i for i,x in enumerate(train.ID.astype(str))}

for fold in range(N_OUTER):
    c=raw[fold]["control"]
    t=raw[fold]["treatment"]

    ids=c["outer_ids"].astype(str)
    idx=np.array([id_to_idx[x] for x in ids],int)

    control_oof[idx]=c["outer_prob"]
    treatment_oof[idx]=t["outer_prob"]

    control_pred[idx]=(c["outer_prob"]>=threshold_control[fold]).astype(np.int8)
    treatment_pred[idx]=(t["outer_prob"]>=threshold_treatment[fold]).astype(np.int8)

    fold_col[idx]=fold

assert np.isfinite(control_oof).all()
assert np.isfinite(treatment_oof).all()
assert (control_pred>=0).all()
assert (treatment_pred>=0).all()

y_all=train[TARGET].to_numpy(int)

pooled={
    "control_f1":f1_pos(y_all,control_pred),
    "treatment_f1":f1_pos(y_all,treatment_pred),
    "control_ap":safe_ap(y_all,control_oof),
    "treatment_ap":safe_ap(y_all,treatment_oof),
    "control_precision":precision_score(y_all,control_pred,zero_division=0),
    "treatment_precision":precision_score(y_all,treatment_pred,zero_division=0),
    "control_recall":recall_score(y_all,control_pred,zero_division=0),
    "treatment_recall":recall_score(y_all,treatment_pred,zero_division=0),
    "true_pos":int(y_all.sum()),
    "control_pred_pos":int(control_pred.sum()),
    "treatment_pred_pos":int(treatment_pred.sum()),
}

pooled["f1_gain"]=pooled["treatment_f1"]-pooled["control_f1"]
pooled["official_gain"]=pooled["f1_gain"]/12.0
pooled["ap_gain"]=pooled["treatment_ap"]-pooled["control_ap"]

print(json.dumps(pooled,indent=2))

script_rows=[]

for script_name in ["native","mixed","no_bengali"]:
    mask=train.script.to_numpy()==script_name

    pos=int(y_all[mask].sum())
    neg=int(mask.sum()-pos)

    c=f1_pos(y_all[mask],control_pred[mask])
    t=f1_pos(y_all[mask],treatment_pred[mask])

    script_rows.append({
        "script":script_name,
        "rows":int(mask.sum()),
        "positive":pos,
        "negative":neg,
        "control_f1":c,
        "treatment_f1":t,
        "f1_gain":t-c,
        "gate_eligible":bool(pos>=30 and neg>=30),
    })

script_report=pd.DataFrame(script_rows)
display(script_report.round(6))
script_report.to_csv(EXPORT_DIR/"v50_script_subgroups.csv",index=False)

oof_df=pd.DataFrame({
    "ID":train.ID.astype(str),
    "outer_fold":fold_col,
    "group_key":train.group_key,
    "script":train.script,
    "y":y_all,
    "control_prob":control_oof,
    "treatment_prob":treatment_oof,
    "control_pred":control_pred,
    "treatment_pred":treatment_pred,
})
oof_df.to_csv(EXPORT_DIR/"v50_oof_predictions.csv",index=False)


In [ ]:

# 18. 10,000 paired group bootstrap, stratified within outer folds

rng=np.random.default_rng(50500)

fold_groups={}

for fold in range(N_OUTER):
    part=train[train.outer_fold==fold]
    d={}
    for g,idxs in part.groupby("group_key").groups.items():
        d[g]=np.asarray(list(idxs),int)
    fold_groups[fold]=d

boot_f1=[]
boot_official=[]

for _ in range(10000):
    idx_parts=[]

    for fold in range(N_OUTER):
        d=fold_groups[fold]
        gs=np.array(list(d.keys()),dtype=object)

        sampled=rng.choice(
            gs,
            size=len(gs),
            replace=True,
        )

        idx_parts.append(
            np.concatenate([d[g] for g in sampled])
        )

    idx=np.concatenate(idx_parts)

    fc=f1_pos(y_all[idx],control_pred[idx])
    ft=f1_pos(y_all[idx],treatment_pred[idx])

    gain=ft-fc

    boot_f1.append(gain)
    boot_official.append(gain/12.0)

boot_f1=np.asarray(boot_f1,float)
boot_official=np.asarray(boot_official,float)

bootstrap={
    "n":10000,
    "mean_f1_gain":float(boot_f1.mean()),
    "f1_p10":float(np.quantile(boot_f1,.10)),
    "f1_p2_5":float(np.quantile(boot_f1,.025)),
    "f1_p97_5":float(np.quantile(boot_f1,.975)),
    "mean_official_gain":float(boot_official.mean()),
    "official_p10":float(np.quantile(boot_official,.10)),
    "official_p2_5":float(np.quantile(boot_official,.025)),
    "official_p97_5":float(np.quantile(boot_official,.975)),
    "positive_fraction":float((boot_official>0).mean()),
}

print(json.dumps(bootstrap,indent=2))


In [ ]:

# 19. Freeze PRIMARY promotion gate before old-500 labels are used

mean_f1_gain=float(fold_report.f1_gain.mean())
mean_official_gain=float(fold_report.official_gain.mean())
wins=int((fold_report.f1_gain>0).sum())
worst_f1=float(fold_report.f1_gain.min())
mean_ap=float(fold_report.ap_gain.mean())
worst_ap=float(fold_report.ap_gain.min())

script_ok=True
script_details={}

for _,r in script_report.iterrows():
    eligible=bool(r.gate_eligible)
    ok=(float(r.f1_gain)>=-0.020) if eligible else None

    script_details[r.script]={
        "eligible":eligible,
        "positive":int(r.positive),
        "negative":int(r.negative),
        "f1_gain":float(r.f1_gain),
        "passes_if_eligible":ok,
    }

    if eligible and not ok:
        script_ok=False

primary_criteria={
    "mean_outer_f1_gain_ge_0.015":mean_f1_gain>=0.015,
    "mean_official_gain_ge_0.00125":mean_official_gain>=0.00125,
    "fold_wins_at_least_2_of_3":wins>=2,
    "worst_fold_f1_ge_minus_0.010":worst_f1>=-0.010,
    "pooled_oof_f1_gain_ge_0.012":pooled["f1_gain"]>=0.012,
    "mean_ap_change_ge_0":mean_ap>=0.0,
    "worst_fold_ap_ge_minus_0.010":worst_ap>=-0.010,
    "bootstrap_positive_fraction_ge_0.90":bootstrap["positive_fraction"]>=0.90,
    "bootstrap_p10_official_gt_0":bootstrap["official_p10"]>0.0,
    "script_stability":script_ok,
    "zero_group_leakage":True,
    "all_six_target_fits_complete":all(
        (RUN_DIR/f"fold{f}_{a}_complete.npz").exists()
        for f in range(N_OUTER)
        for a in ["control","treatment"]
    ),
}

PRIMARY_PASS=bool(all(primary_criteria.values()))

primary_summary={
    "mean_outer_f1_gain":mean_f1_gain,
    "mean_official_gain":mean_official_gain,
    "fold_wins":wins,
    "worst_fold_f1_gain":worst_f1,
    "pooled":pooled,
    "mean_ap_gain":mean_ap,
    "worst_fold_ap_gain":worst_ap,
    "script_details":script_details,
    "bootstrap":bootstrap,
    "criteria":primary_criteria,
    "PRIMARY_PASS":PRIMARY_PASS,
}

print("PRIMARY RESULTS FROZEN.")
print(json.dumps(primary_summary,indent=2))

with open(EXPORT_DIR/"v50_primary_summary.json","w") as f:
    json.dump(primary_summary,f,indent=2)


In [ ]:

# 20. Historical 500-row final VETO + save test margins
#
# Primary configuration/results are already frozen.
# No model selection is done on this split.

control_margin500=[]
treatment_margin500=[]
control_margin_test=[]
treatment_margin_test=[]

for fold in range(N_OUTER):
    c=raw[fold]["control"]
    t=raw[fold]["treatment"]

    control_margin500.append(
        calibrated_margin(c["valid500_prob"],threshold_control[fold])
    )
    treatment_margin500.append(
        calibrated_margin(t["valid500_prob"],threshold_treatment[fold])
    )

    control_margin_test.append(
        calibrated_margin(c["test_prob"],threshold_control[fold])
    )
    treatment_margin_test.append(
        calibrated_margin(t["test_prob"],threshold_treatment[fold])
    )

mean_c500=np.mean(control_margin500,axis=0)
mean_t500=np.mean(treatment_margin500,axis=0)

mean_ctest=np.mean(control_margin_test,axis=0)
mean_ttest=np.mean(treatment_margin_test,axis=0)

y500=valid500[TARGET].to_numpy(int)

control_pred500=(mean_c500>=0).astype(int)
treatment_pred500=(mean_t500>=0).astype(int)

# Frozen V38 identity rule:
# 0.70 * V2 + 0.30 * oldb snapshot 2, threshold 0.3814
identity_j = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
].index("identity_attack")

v38_score500=(
    .70*v2v.identity_attack_prob.to_numpy(float)
    +.30*oldb["valid"][2,:,identity_j]
)

v38_pred500=(v38_score500>=0.3814).astype(int)

secondary={
    "v38_identity_f1":f1_pos(y500,v38_pred500),
    "control_identity_f1":f1_pos(y500,control_pred500),
    "treatment_identity_f1":f1_pos(y500,treatment_pred500),
    "v38_identity_ap":safe_ap(y500,v38_score500),
    "control_identity_ap":safe_ap(y500,mean_c500),
    "treatment_identity_ap":safe_ap(y500,mean_t500),
}

secondary["veto_treatment_f1_not_below_v38"] = bool(
    secondary["treatment_identity_f1"] >= secondary["v38_identity_f1"]
)

VETO_PASS=bool(secondary["veto_treatment_f1_not_below_v38"])
ADVANCE=bool(PRIMARY_PASS and VETO_PASS)

print("SECONDARY VETO ONLY:")
print(json.dumps(secondary,indent=2))
print("PRIMARY_PASS =",PRIMARY_PASS)
print("VETO_PASS    =",VETO_PASS)
print("ADVANCE      =",ADVANCE)


In [ ]:

# 21. Package V50 results
#
# No Kaggle challenger submission is created.

prob500=pd.DataFrame({
    "ID":valid500.ID.astype(str),
    "v38_identity_score":v38_score500,
    "v50_control_calibrated_margin":mean_c500,
    "v50_treatment_calibrated_margin":mean_t500,
})

probtest=pd.DataFrame({
    "ID":test.ID.astype(str),
    "v50_control_calibrated_margin":mean_ctest,
    "v50_treatment_calibrated_margin":mean_ttest,
})

for fold in range(N_OUTER):
    prob500[f"control_fold{fold}_prob"]=raw[fold]["control"]["valid500_prob"]
    prob500[f"treatment_fold{fold}_prob"]=raw[fold]["treatment"]["valid500_prob"]

    probtest[f"control_fold{fold}_prob"]=raw[fold]["control"]["test_prob"]
    probtest[f"treatment_fold{fold}_prob"]=raw[fold]["treatment"]["test_prob"]

prob500.to_csv(
    EXPORT_DIR/"v50_validation500_probabilities.csv",
    index=False
)
probtest.to_csv(
    EXPORT_DIR/"v50_test_probabilities.csv",
    index=False
)

champ.to_csv(
    EXPORT_DIR/"submission_v38_PROTECTED.csv",
    index=False
)

# Assert the six non-identity V38 columns are simply preserved artifacts;
# V50 generated no submission and changed no protected output.
protected_cols=[
    "is_toxic","implicit_toxicity","sarcasm",
    "contains_profanity","threatening","sexual_content"
]
assert champ[protected_cols].equals(
    pd.read_csv(EXPORT_DIR/"submission_v38_PROTECTED.csv")[protected_cols]
)

import transformers

environment={
    "python":platform.python_version(),
    "torch":torch.__version__,
    "transformers":transformers.__version__,
    "sklearn":sklearn.__version__,
    "pandas":pd.__version__,
    "numpy":np.__version__,
    "gpu":torch.cuda.get_device_name(0),
}

summary={
    "protected_public_lb":PROTECTED_PUBLIC_LB,
    "minimum_public_lb_goal":MIN_PUBLIC_LB_GOAL,
    "main_public_lb_goal":MAIN_PUBLIC_LB_GOAL,
    "experiment":"BanTH seven-target intermediate adaptation -> competition identity_attack",
    "model_repo":MODEL_REPO,
    "model_revision":MODEL_REVISION,
    "model_weight_sha256":EXPECTED_MODEL_WEIGHT_SHA256,
    "banth_repo":BANTH_REPO,
    "banth_revision":BANTH_REVISION,
    "banth_train_sha256":banth_file_hash,
    "banth_retained_rows":int(len(banth)),
    "banth_positive_groups":positive_group_counts,
    "primary":primary_summary,
    "secondary_old500_veto":secondary,
    "PRIMARY_PASS":PRIMARY_PASS,
    "VETO_PASS":VETO_PASS,
    "ADVANCE":ADVANCE,
    "decision":(
        "ADVANCE to a separate conservative V38 identity integration check; "
        "do not yet call V50 the champion."
        if ADVANCE else
        "REJECT V50; preserve V38 = 0.608."
    ),
    "environment":environment,
    "data_manifest":supplied_manifest,
}

with open(EXPORT_DIR/"v50_summary.json","w") as f:
    json.dump(summary,f,indent=2)

with open(EXPORT_DIR/"v50_config.json","w") as f:
    json.dump({
        **CONFIG_CORE,
        "config_fingerprint":CONFIG_FP,
        "source_positive_weights":SOURCE_POS_WEIGHT.tolist(),
        "source_positive_groups":positive_group_counts,
        "overlap_rules":{
            "raw_exact":True,
            "canonical_exact":"NFC + casefold + whitespace + zero-width removal",
            "near_jaccard":NEAR_JACCARD,
            "near_length_ratio":NEAR_LENGTH_RATIO,
            "minimum_near_chars":MIN_NEAR_CHARS,
            "alternate_views_for_exclusion_only":["bangla","english"],
        },
    },f,indent=2)

decision_text=f'''
V50 BANTH IDENTITY TRANSFER

Protected champion: V38 = {PROTECTED_PUBLIC_LB}
Minimum leaderboard goal: {MIN_PUBLIC_LB_GOAL}
Main target: {MAIN_PUBLIC_LB_GOAL}

PRIMARY_PASS = {PRIMARY_PASS}
VETO_PASS    = {VETO_PASS}
ADVANCE      = {ADVANCE}

{summary["decision"]}

No Kaggle challenger submission was generated.
Upload v50_results.zip to ChatGPT for review.
'''.strip()

(EXPORT_DIR/"README_DECISION.txt").write_text(decision_text)

zip_path=Path("/content/v50_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print(decision_text)
print()
print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
